In [ ]:
#!/usr/bin/env python3
"""
volume_to_weight_train_gridsearch.py

Modified version:
 - Uses 3 rounds of Grid Search CV (instead of Bayesian optimization).
 - Each round runs 5 epochs to quickly evaluate hyperparameters.
 - Each round narrows the search space around the previous best.
 - Final training uses full epochs (default = 50).
 - After training, prints MSE, MAE, and R² on the test set.
"""

import os
import math
import csv
import random
from typing import List, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ----------------------------
# Utilities: Otsu thresholding
# ----------------------------
def otsu_threshold_from_array(gray: np.ndarray) -> int:
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
    total = hist.sum()
    if total == 0:
        return 0
    prob = hist.astype(np.float64) / total
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * np.arange(256))
    mu_total = mu[-1]
    denom = omega * (1.0 - omega) + 1e-12
    sigma_b2 = (mu_total * omega - mu) ** 2 / denom
    sigma_b2[omega == 0] = 0
    sigma_b2[omega == 1] = 0
    return int(np.argmax(sigma_b2))

# ----------------------------
# Mask / bbox / volume utils
# ----------------------------
def image_to_mask(img_path: str) -> np.ndarray:
    img = Image.open(img_path).convert("L")
    arr = np.array(img)
    t = otsu_threshold_from_array(arr)
    mask = (arr > t).astype(np.uint8)
    return mask

def bbox_from_mask(mask: np.ndarray) -> Tuple[int,int,int,int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0,0,0,0)
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def avg_row_width(mask: np.ndarray, y: int, minx: int, maxx: int, half_window: int = 1) -> float:
    H, W = mask.shape
    ys = range(max(0, y-half_window), min(H, y+half_window+1))
    widths = [int(np.sum(mask[yy, minx:maxx+1] > 0)) for yy in ys]
    return float(np.mean(widths)) if widths else 0.0

def estimate_volume_from_mask(mask: np.ndarray, N_slices: int, bbox: Tuple[int,int,int,int], boundary_window: int = 1) -> float:
    minx, miny, maxx, maxy = bbox
    if minx == maxx and miny == maxy:
        return 0.0

    L = maxy - miny + 1
    slice_h = float(L) / float(N_slices)

    def area_at_row(yf: float) -> float:
        y = int(round(yf))
        y = max(miny, min(maxy, y))
        w = avg_row_width(mask, y, minx, maxx, boundary_window)
        return (math.pi / 4.0) * (w ** 2)

    total_vol = 0.0
    for i in range(N_slices):
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        h = bottom_f - top_f
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            V = (h / 3.0) * A_top
        else:
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        total_vol += V
    return total_vol

# ----------------------------
# CSV loader
# ----------------------------
def load_csv_pairs(csv_file: str, img_root: str = "") -> List[Tuple[str, float]]:
    pairs = []
    missing = []
    img_root_norm = os.path.normpath(img_root) if img_root else ""

    with open(csv_file, "r", newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if header and len(header) >= 2:
            has_header = "image" in header[0].lower() or "weight" in header[1].lower()
        else:
            has_header = False

        def resolve_path(image_name: str) -> str:
            image_name = image_name.strip().strip('"').strip("'")
            pth = os.path.join(img_root_norm, image_name)
            if os.path.exists(pth):
                return os.path.normpath(pth)
            return ""

        if not has_header:
            try:
                img_p = resolve_path(header[0])
                wt = float(header[1])
                if img_p:
                    pairs.append((img_p, wt))
                else:
                    missing.append(header[0])
            except:
                missing.append(header[0])

        for row in reader:
            if len(row) < 2:
                continue
            img_name = row[0]
            try:
                wt = float(row[1])
            except:
                try:
                    wt = float(row[1].strip())
                except:
                    missing.append(img_name)
                    continue
            img_p = resolve_path(img_name)
            if img_p:
                pairs.append((img_p, wt))
            else:
                missing.append(img_name)

    if missing:
        print(f"[warning] {len(missing)} image paths in CSV could not be resolved - skipped.")
    print(f"[info] Loaded {len(pairs)} valid image pairs from '{csv_file}'.")
    return pairs

# ----------------------------
# Features & model
# ----------------------------
class SimpleScaler:
    def fit(self, X): 
        self.mean_, self.std_ = np.mean(X,0,keepdims=True), np.std(X,0,keepdims=True)
        self.std_[self.std_==0]=1
    def transform(self, X): return (X - self.mean_) / self.std_
    def fit_transform(self, X): self.fit(X); return self.transform(X)

def poly_features(vols: np.ndarray, degree: int) -> np.ndarray:
    vols = np.array(vols).reshape(-1,1)
    return np.column_stack([vols[:,0]**d for d in range(1, degree+1)])

class PolyRegModel(nn.Module):
    def __init__(self, in_features): 
        super().__init__(); self.lin = nn.Linear(in_features, 1)
    def forward(self, x): return self.lin(x).squeeze(1)

# ----------------------------
# Training routine
# ----------------------------
def train_one_model(X_train, y_train, X_val, y_val, degree, lr, alpha, l1_ratio, batch_size, epochs, device):
    X_train_f = poly_features(X_train.reshape(-1,1), degree)
    X_val_f = poly_features(X_val.reshape(-1,1), degree)
    scaler = SimpleScaler()
    X_train_s = scaler.fit_transform(X_train_f)
    X_val_s = scaler.transform(X_val_f)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_s,dtype=torch.float32), torch.tensor(y_train,dtype=torch.float32)),
                              batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_s,dtype=torch.float32), torch.tensor(y_val,dtype=torch.float32)),
                            batch_size=max(1,batch_size//2), shuffle=False)

    model = PolyRegModel(degree).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state = float('inf'), None
    for _ in range(epochs):
        model.train()
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            mse = ((preds - yb)**2).mean()
            l1 = sum(p.abs().sum() for p in model.parameters())
            l2 = sum((p**2).sum() for p in model.parameters())
            loss = mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for xb,yb in val_loader:
                xb,yb = xb.to(device), yb.to(device)
                preds = model(xb)
                mse = ((preds - yb)**2).mean()
                l1 = sum(p.abs().sum() for p in model.parameters())
                l2 = sum((p**2).sum() for p in model.parameters())
                val_loss += (mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)).item()
            val_loss /= len(val_loader)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, scaler, best_loss

# ----------------------------
# Metrics computation
# ----------------------------
def compute_metrics_and_preds(model, scaler, X, y, degree, device):
    X_f = poly_features(np.array(X).reshape(-1,1), degree)
    X_s = scaler.transform(X_f)
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_s,dtype=torch.float32).to(device)).cpu().numpy()
    mse = ((preds - y)**2).mean()
    mae = np.mean(np.abs(preds - y))
    ss_res = ((y - preds) ** 2).sum()
    ss_tot = ((y - np.mean(y)) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {"mse": mse, "mae": mae, "r2": r2}, preds

# ----------------------------
# Grid search
# ----------------------------
def grid_search_cv(X_train, y_train, X_val, y_val, device, batch_size, rounds=3, epochs_per_round=5):
    # Initial broad grid
    param_grid = {
        "lr": [1e-3, 1e-2, 1e-1],
        "alpha": [1e-6, 1e-4, 1e-2],
        "l1_ratio": [0.0, 0.5, 1.0],
        "degree": [1, 2, 3, 4, 5],
        "N_slices": [8, 12, 16]
    }

    best_params = None
    for r in range(rounds):
        print(f"\n--- Grid Search Round {r+1}/{rounds} ---")
        best_loss = float("inf")
        for lr in param_grid["lr"]:
            for alpha in param_grid["alpha"]:
                for l1_ratio in param_grid["l1_ratio"]:
                    for degree in param_grid["degree"]:
                        for N_slices in param_grid["N_slices"]:
                            model, scaler, vloss = train_one_model(
                                X_train, y_train, X_val, y_val,
                                degree, lr, alpha, l1_ratio,
                                batch_size, epochs_per_round, device
                            )
                            if vloss < best_loss:
                                best_loss = vloss
                                best_params = {
                                    "lr": lr,
                                    "alpha": alpha,
                                    "l1_ratio": l1_ratio,
                                    "degree": degree,
                                    "N_slices": N_slices
                                }
        print("Best so far:", best_params, "loss:", best_loss)

        # Narrow grid for next round
        def narrow(values, best, factor=2.0):
            if isinstance(values[0], float):
                return sorted(set([best/factor, best, best*factor]))
            else:
                return sorted(set([max(1,best-1), best, best+1]))

        param_grid = {
            "lr": narrow([0], best_params["lr"]),
            "alpha": narrow([0], best_params["alpha"]),
            "l1_ratio": [max(0.0,best_params["l1_ratio"]-0.25),
                         best_params["l1_ratio"],
                         min(1.0,best_params["l1_ratio"]+0.25)],
            "degree": narrow([0], best_params["degree"]),
            "N_slices": narrow([0], best_params["N_slices"])
        }

    print("\n=== Final Best Params ===")
    print(best_params)
    return best_params

# ----------------------------
# Main pipeline
# ----------------------------
def run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max, *,
                 epochs=50, batch_size=8, device="auto", seed=42, save_test_preds=False):

    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    pairs = load_csv_pairs(csv_file, img_root)
    weights = np.array([w for _,w in pairs])
    n = len(pairs)
    idx = np.arange(n); np.random.shuffle(idx)
    val_n, test_n = max(1,int(0.2*n)), max(1,int(0.1*n))
    val_idx, test_idx, train_idx = idx[:val_n], idx[val_n:val_n+test_n], idx[val_n+test_n:]

    # Precompute volumes for initial N_slices range
    vols_dict = {}
    for N_slices in range(N_slices_min, N_slices_max+1):
        vols = []
        for p in pairs:
            mask = image_to_mask(p[0])
            bbox = bbox_from_mask(mask)
            vols.append(estimate_volume_from_mask(mask, N_slices, bbox))
        vols_dict[N_slices] = np.array(vols).reshape(-1,1)

    # Grid search with validation split
    best_params = grid_search_cv(
        vols_dict[N_slices_min], weights[train_idx], 
        vols_dict[N_slices_min][val_idx], weights[val_idx],
        device, batch_size
    )

    # Final volumes with chosen N_slices
    vols = vols_dict.get(best_params["N_slices"])
    if vols is None:
        vols = vols_dict[N_slices_min]

    Xtr = np.delete(vols, val_idx, axis=0)
    ytr = np.delete(weights, val_idx)
    Xte, yte = vols[test_idx], weights[test_idx]

    # Train final model
    model, scaler, _ = train_one_model(Xtr, ytr, Xte, yte, int(best_params["degree"]),
                                       best_params["lr"], best_params["alpha"], best_params["l1_ratio"],
                                       batch_size, epochs, device)

    torch.save({"state": model.state_dict(),
                "scaler_mean": scaler.mean_,
                "scaler_std": scaler.std_,
                "params": best_params}, out_model)
    print(f"Model saved to {out_model}")

    # ---- Compute test metrics ----
    metrics, preds = compute_metrics_and_preds(model, scaler, Xte, yte, int(best_params["degree"]), device)
    print("\n=== Test Set Metrics ===")
    print(f"MSE: {metrics['mse']:.4f}")
    print(f"MAE: {metrics['mae']:.4f}")
    print(f"R² : {metrics['r2']:.4f}")

    if save_test_preds:
        out_csv = os.path.splitext(out_model)[0] + "_test_preds.csv"
        with open(out_csv, "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Image", "True_Weight", "Pred_Weight"])
            for i, idx in enumerate(test_idx):
                writer.writerow([pairs[idx][0], weights[idx], preds[i]])
        print(f"Test predictions saved to {out_csv}")

# ----------------------------
# Hard-coded config
# ----------------------------
if __name__=="__main__":
    img_root = "Tomato-Images/Tomato/"   # folder containing images referenced in CSV
    csv_file = "tomato.csv"  # CSV with columns: Image Name, Weight, View, Tomato ID
    out_model = "tomato_volume_model_gridsearch.pth"
    N_slices_min = 10
    N_slices_max = 15

    epochs = 50
    batch_size = 8
    device = "auto"
    seed = 42
    save_test_preds = True

    run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max,
                 epochs=epochs, batch_size=batch_size, device=device, seed=seed,
                 save_test_preds=save_test_preds)